# Category Classification — Feature Pipeline

**Responsibility:** raw CSV → cleaned, engineered feature rows uploaded to the Hopsworks Feature Store. No model-specific logic lives here — only data contracts.

| Output | Description |
|---|---|
| Hopsworks Feature Group `category_features` (v1) | One row per observation with `split` column and `row_id` primary key |
| `category_metadata.json` | Label maps and feature-column lists (no sklearn objects) |

> **Prerequisites:** none — this is the first stage.
> **Next stage:** run `category_classification_t.ipynb` (Training Pipeline).

## Environment Setup

This section ensures the notebook environment is ready for execution.
It checks that required packages are installed (see `requirements.txt` or `README.md`),
sets the working directory, and verifies access to local data paths.
All downstream cells depend on this foundation being in place.

**Inputs:** local Python environment, `README.md` for dependency list.  
**Outputs:** verified runtime ready for imports.

## Shared Imports & Configuration

This cell loads the core libraries used across the entire Feature Pipeline:
standard-library modules for I/O and reproducibility, NumPy/Pandas for data manipulation,
Matplotlib/Seaborn for diagnostics, and the Hopsworks SDK for Feature Store interaction.
It also suppresses warnings and configures display defaults so notebook output stays readable.

**Outputs:** imported namespaces (`pd`, `np`, `hopsworks`, etc.) and global display settings.

In [ ]:
import json          # serialising pipeline metadata (FP-6)
import os            # reading environment-variable overrides for Hopsworks secrets
import random        # seeding Python's built-in RNG for reproducibility
import warnings      # suppressing noisy third-party deprecation warnings
from pathlib import Path  # object-oriented filesystem paths

import numpy as np   # numerical arrays and vectorised operations
import pandas as pd  # tabular data manipulation and feature engineering

import matplotlib.pyplot as plt  # plotting backend
import seaborn as sns            # statistical visualisations on top of matplotlib

import hopsworks     # Hopsworks Feature Store SDK (connect, create FG/FV, ingest)
import tomllib      # Python 3.11+ standard-library parser for TOML (PEP 680)
from category_classification import (
    parse_first, parse_count,       # parse list-like strings from DSA raw data
    merge_arrow_schema,             # normalise schema across chunked parquet files
    read_data_file,                 # unified CSV / parquet / gz reader
    fp3_platform_worker,            # parallel worker for 3-hour rolling features
)
from fa0_data_acquisition import setup_ui
from fa0_data_acquisition import run_acquisition
from tqdm.auto import tqdm as _tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed


# Silence deprecation and future warnings so the notebook log stays focused on pipeline output.
warnings.filterwarnings("ignore")

# Set default figure size and font size for all plots generated in FP-2 and later diagnostics.
plt.rcParams.update({"figure.figsize": (14, 6), "figure.dpi": 100, "font.size": 11})
sns.set_style("whitegrid")

# Prevent pandas from truncating wide DataFrames when we inspect engineered features.
pd.set_option("display.max_columns", 50)

# Quick sanity check: if this line prints, the environment has all required packages.
print("Imports OK.")

## TOML Configuration & Distribution Helper

The notebook reads a single TOML file (`category_classification_fti.toml`) that acts as the
source of truth for all paths, hyperparameters, domain constants, and Hopsworks credentials.
Centralising configuration makes the pipeline portable across local, CI, and production
environments without code changes.  We also import helper functions shared with the Training
Pipeline (`category_classification.py`) so data-parsing logic is not duplicated.

**Inputs:** `category_classification_fti.toml`, `category_classification.py`.  
**Outputs:** `cfg` (nested dict) and shared helper callables (`parse_first`, `parse_count`, etc.).

In [ ]:
# Shared helpers used by both the Feature and Training pipelines.
# (imported in the first cell above)

# Resolve the config path relative to the notebook working directory.
CFG_PATH = Path("category_classification_fti.toml")

# tomllib requires a binary file object, so we open in "rb" mode.
with open(CFG_PATH, "rb") as f:
    cfg = tomllib.load(f)

print(f"Config loaded from {CFG_PATH}")

## Path, Constant & Service Definitions

Resolve all file-system paths from config, pin the random seed, and wire up Hopsworks connection parameters. Every downstream cell reads from these constants — edit the TOML to change them.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# Resolve every artifact path from the TOML `paths` section so downstream cells
# can read and write without hard-coding locations.
_paths = cfg["paths"]
MODEL_ARTIFACT    = Path(_paths["model_artifact"])
MODEL_EXPORT_DIR  = Path(_paths["model_export_dir"])
CV_RESULTS_PATH   = Path(_paths["cv_results"])
PREDICTIONS_PATH  = Path(_paths["predictions"])
METADATA_PATH     = Path(_paths["metadata"])

# ── Reproducibility ────────────────────────────────────────────────────────────
# Pin random seeds and CV fold count so splits and model training are deterministic.
_repr = cfg["reproducibility"]
RANDOM_STATE = _repr["random_state"]
N_CV_FOLDS   = _repr["n_cv_folds"]
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# ── Domain constants ───────────────────────────────────────────────────────────
# Prefixes and column names that define the raw data schema and target collapsing rules.
_domain = cfg["domain"]
OTHER_CAT  = _domain["other_category"]   # catch-all label used when classes are rare
PREFIX     = _domain["prefix"]           # string stripped from raw category names

TEXT_COL   = _domain["text_col"]         # primary text feature (e.g. content ground)
TEXT_COL_2 = _domain["text_col_2"]       # secondary text feature (e.g. explanation)
TEXT_COL_3 = _domain["text_col_3"]       # tertiary text feature (e.g. decision facts)
HIGH_CARD  = cfg["features"]["categorical"]["encoding"]["high_cardinality"]

# ── Hopsworks ─────────────────────────────────────────────────────────────────
# Connection parameters.  Environment variables take precedence over TOML so secrets
# never need to be committed to version control.
_hw = cfg["hopsworks"]
HOPSWORKS_HOST    = os.environ.get("HOPSWORKS_HOST",    _hw["host"])
HOPSWORKS_PROJECT = os.environ.get("HOPSWORKS_PROJECT", _hw["project"])
HOPSWORKS_API_KEY = os.environ.get("HOPSWORKS_API_KEY", _hw["api_key"])

# Feature Group / Feature View names and versions.  Keeping them in config allows
# multiple parallel experiments without name collisions.
FG_NAME           = _hw["feature_group"]["name"]
FG_VERSION        = _hw["feature_group"]["version"]
FG_TEXT_1_NAME    = _hw.get("feature_group_text_1", {}).get("name",    FG_NAME + "_text_1")
FG_TEXT_1_VERSION = _hw.get("feature_group_text_1", {}).get("version", 1)
FG_TEXT_2_NAME    = _hw.get("feature_group_text_2", {}).get("name",    FG_NAME + "_text_2")
FG_TEXT_2_VERSION = _hw.get("feature_group_text_2", {}).get("version", 1)
FG_TEXT_3_NAME    = _hw.get("feature_group_text_3", {}).get("name",    FG_NAME + "_text_3")
FG_TEXT_3_VERSION = _hw.get("feature_group_text_3", {}).get("version", 1)
FV_NAME           = _hw.get("feature_view", {}).get("name",    FG_NAME + "_view")
FV_VERSION        = _hw.get("feature_view", {}).get("version", 1)
MODEL_NAME        = _hw["model_registry"]["name"]

print(f"Metadata  : {METADATA_PATH}")
print(f"Model     : {MODEL_ARTIFACT}")
print(f"Hopsworks : {HOPSWORKS_HOST}  project={HOPSWORKS_PROJECT}")
print(f"  FG (struct) : {FG_NAME} v{FG_VERSION}")
print(f"  FG (text 1) : {FG_TEXT_1_NAME} v{FG_TEXT_1_VERSION}")
print(f"  FG (text 2) : {FG_TEXT_2_NAME} v{FG_TEXT_2_VERSION}")
print(f"  FG (text 3) : {FG_TEXT_3_NAME} v{FG_TEXT_3_VERSION}")
print(f"  FV          : {FV_NAME} v{FV_VERSION}")
print(f"  Model       : {MODEL_NAME}")

## Hopsworks — Project Setup

Connect to the Hopsworks cluster and ensure the project exists. On first run with
Hopsworks Serverless the project is auto-provisioned; on-premise deployments may
require manual project creation via the UI (see commented snippet in the cell below).

`project` and `fs` are module-level variables reused by FP-5 (feature group), FP-5.5
(feature view), and the Model Registry upload in `_t.ipynb`.

In [ ]:
# ── Connect and create / get project ──────────────────────────────────────────
# hopsworks.login() auto-provisions a project on first use in Hopsworks Serverless;
# on subsequent calls it simply reconnects.  On-premise deployments that do not
# auto-create projects should uncomment the block below and use connection().create_project().
#
#   import hopsworks
#   _conn = hopsworks.connection(host=HOPSWORKS_HOST, api_key_value=HOPSWORKS_API_KEY)
#   project = _conn.create_project(HOPSWORKS_PROJECT, description=cfg["project"]["description"])

# Authenticate and obtain a project handle.  This object is reused by FP-4 (feature groups),
# FP-5 (feature view), and the training pipeline when registering models.
project = hopsworks.login(
    host=HOPSWORKS_HOST,
    project=HOPSWORKS_PROJECT,
    api_key_value=HOPSWORKS_API_KEY,
)

# Get the project's Feature Store instance — the central API for creating and querying
# feature groups and views.
fs = project.get_feature_store()
print(f"Connected  : {HOPSWORKS_HOST}  →  project '{project.name}'  (id={project.id})")
print(f"Feature Store : {fs.name}")

---
# Part 1 — Feature Pipeline

**Responsibility:** raw CSV → cleaned, engineered feature rows uploaded to the Hopsworks
Feature Store. No model-specific logic lives here — only data contracts.

**Output:**
- Hopsworks Feature Group `category_features` (v1) — one row per observation with a
  `split` column (`"train"` / `"test"`) and a `row_id` primary key.
- `category_metadata.json` — label maps, feature-column lists (no sklearn objects).

## FA-0 · Data Acquisition

> The code for this step is extracted into `fa0_data_acquisition.py` for brevity.
> See that module for the full implementation.

Two input modes — select via the radio button below:

| Mode | Description |
|---|---|
| **⬇ Download** | Streams an n-th-row sample from the CloudFront daily parquet zips, one day at a time (constant peak RAM ≈ one chunk). Saved to the **Folder** path below. |
| **📂 Load from folder** | Recursively scans the **Folder** path for `.csv`, `.csv.gz`, and `.parquet` files at any nesting depth. Select one or more files; each file is read, optionally sampled, and written to a single output parquet one at a time — peak RAM ≈ one file. |

The **Folder** widget is shared: downloaded files land in that folder, and the file scanner reads from it. After a download completes, switch to Load mode, hit **↺ Refresh**, and the new file will appear in the list.

In [ ]:
# FA-0 widget UI — extracted to fa0_data_acquisition.py to keep the notebook focused
# on pipeline orchestration rather than widget boilerplate.

# Render the interactive ipywidgets (radio buttons, file pickers, progress bars).
# The returned namespace object holds the widget state needed by the execution cell below.
_fa0_ns = setup_ui(cfg)


### FA-0 · Execute Acquisition

Run the configured mode: **Download** streams chunked daily parquet zips from CloudFront
and writes a sampled parquet to the configured folder; **Load** reads local CSV / parquet
files and writes a merged, optionally sampled parquet. Either way, `OUT_PATH` points to
the resulting file consumed by FP-1.

> The implementation of this step is extracted into `fa0_data_acquisition.py`.

In [ ]:
# FA-0 execution — extracted to fa0_data_acquisition.py for maintainability.

# Run the selected acquisition mode (Download or Load) and return the path to the
# resulting parquet file.  FP-1 will verify this path exists before attempting to read.
OUT_PATH = run_acquisition(cfg, merge_arrow_schema, read_data_file, _fa0_ns)


## FP-1 · Data Loading & Feature Engineering

FP-1 is the core transformation stage of the Feature Pipeline.  It reads the raw parquet
produced by FA-0, then applies 13 sequential cleaning and engineering steps.  The result is
a fully prepared DataFrame (`df`) containing parsed categorical lists, date-derived features,
decision-action flags, text-length proxies, and placeholder columns for rolling-window
statistics that will be recomputed chronologically in FP-3.

**Inputs:** `OUT_PATH` (parquet from FA-0).  
**Outputs:** mutated `df` (engineered features), `_has_subday` boolean (used by FP-3).

### FP-1 · Processing

Executes 13 sequential feature-engineering steps on the loaded parquet:
parse list-columns (`content_type`, `territorial_scope`, `decision_visibility`),
compute decision-action flags, extract date parts and content-application lag,
derive a text-length proxy, check timestamp precision, set rolling-feature defaults,
fill missing text, and emit diagnostic statistics.

**Inputs:** `OUT_PATH` parquet written by FA-0.  
**Outputs:** cleaned `df` with new columns; `_has_subday` flag forwarded to FP-3.

In [ ]:

# parse_first, parse_count imported from category_classification

# Total number of engineering steps; used to drive the progress bar.
_FP1_STEPS = 13
with _tqdm(total=_FP1_STEPS, desc="FP-1", unit="step") as _pbar:

    # ── 1 · Load ──────────────────────────────────────────────────────────────
    # Read the parquet written by FA-0.  If the acquisition cell hasn't been
    # run, OUT_PATH won't exist and we fail fast with a clear message.
    _pbar.set_description("load")
    if "OUT_PATH" in vars() and OUT_PATH and Path(OUT_PATH).exists():
        _data_source = Path(OUT_PATH)
        df = pd.read_parquet(_data_source)
    else:
        raise RuntimeError(
            "No parquet output from FA-0 — re-run the data acquisition cell above."
        )
    _pbar.write(f"Loaded {len(df):,} rows × {df.shape[1]} columns  ← {_data_source}")
    _pbar.update(1)

    # ── 2 · content_type features → drop source ───────────────────────────────
    # content_type arrives as a list-like string (e.g. "[VIDEO, AUDIO]").
    # We extract the first element (primary type) and the count (how many types)
    # because the raw list string is useless to tree-based models.
    _pbar.set_description("content_type")
    df["content_type_primary"] = df["content_type"].apply(parse_first)
    df["content_type_count"]   = df["content_type"].apply(parse_count).astype(np.int32)
    df.drop(columns=["content_type"], inplace=True, errors="ignore")
    _pbar.update(1)

    # ── 3 · territorial_scope features → drop source ──────────────────────────
    # Same pattern as content_type: scope is a list string.  Count captures
    # cross-border complexity; primary captures the dominant jurisdiction.
    _pbar.set_description("territorial_scope")
    df["territorial_scope_count"]   = df["territorial_scope"].apply(parse_count).astype(np.int32)
    df["territorial_scope_primary"] = df["territorial_scope"].apply(parse_first)
    df.drop(columns=["territorial_scope"], inplace=True, errors="ignore")
    _pbar.update(1)

    # ── 4 · visibility features (keep source for step 5) ──────────────────────
    # decision_visibility is needed again in step 5 to compute total_decision_actions,
    # so we only extract primary/count here and defer dropping until step 5.
    _pbar.set_description("decision_visibility")
    df["decision_visibility_primary"] = df["decision_visibility"].apply(parse_first)
    df["visibility_decision_count"]   = df["decision_visibility"].apply(parse_count).astype(np.int32)
    _pbar.update(1)

    # ── 5 · decision action flags → drop decision_visibility ──────────────────
    # Transform the four raw decision columns (visibility, monetary, provision, account)
    # into binary flags and a composite multi-action indicator.  This turns sparse,
    # semi-structured text into dense numeric features that any classifier can consume.
    _pbar.set_description("decision flags")
    df["has_monetary_decision"]  = df["decision_monetary"].notna().astype(np.int8)
    df["has_provision_decision"] = df["decision_provision"].notna().astype(np.int8)
    df["has_account_decision"]   = df["decision_account"].notna().astype(np.int8)
    df["total_decision_actions"] = (
        df["decision_visibility"].notna().astype(np.int8)
        + df["has_monetary_decision"]
        + df["has_provision_decision"]
        + df["has_account_decision"]
    ).astype(np.int8)
    df["is_multi_action_decision"] = (df["total_decision_actions"] > 1).astype(np.int8)
    df.drop(columns=["decision_visibility"], inplace=True, errors="ignore")
    _pbar.update(1)

    # ── 6 · app_year, app_month ───────────────────────────────────────────────
    # application_date is a raw ISO string.  Extracting year and month gives
    # the model a strong seasonal signal (e.g. holiday enforcement spikes).
    _pbar.set_description("app date")
    app_dt = pd.to_datetime(df["application_date"], errors="coerce")
    df["app_year"]  = app_dt.dt.year.astype("Int16")
    df["app_month"] = app_dt.dt.month.astype("Int8")
    _pbar.update(1)

    # ── 7 · content_year ──────────────────────────────────────────────────────
    # content_date is when the offending material was posted.  The year alone
    # captures content vintage without leaking the exact timestamp.
    _pbar.set_description("content_year")
    cnt_dt = pd.to_datetime(df["content_date"], errors="coerce")
    df["content_year"] = cnt_dt.dt.year.astype("Int16")
    _pbar.update(1)

    # ── 8 · lag days → drop content_date, free datetime arrays ────────────────
    # How long after creation was the decision applied?  Long lags may indicate
    # backlog or complex cases.  We drop the raw dates after extraction to save RAM.
    _pbar.set_description("lag days")
    df["content_application_lag_days"] = (app_dt - cnt_dt).dt.days.astype("Int32")
    df.drop(columns=["content_date"], inplace=True, errors="ignore")
    del app_dt, cnt_dt
    _pbar.update(1)

    # ── 9 · text length proxy; only drop decision_facts if it isn't a text column ─
    # decision_facts_len serves as a cheap proxy for case complexity (longer facts
    # = more nuanced reasoning).  We keep the raw text if it will later be stored
    # in a text feature group; otherwise we drop it to reduce memory.
    _pbar.set_description("facts length")
    df["decision_facts_len"] = df["decision_facts"].fillna("").str.len().astype(np.int32)
    if "decision_facts" not in {TEXT_COL, TEXT_COL_2, TEXT_COL_3}:
        df.drop(columns=["decision_facts"], inplace=True, errors="ignore")
    _pbar.update(1)

    # ── 10 · timestamp precision check (result used by FP-3) ──────────────────
    # Rolling window features only make sense if created_at has sub-day precision.
    # If every timestamp is exactly midnight, we skip expensive rolling computation
    # and leave the placeholder defaults from step 11.
    _pbar.set_description("timestamp")
    _ts_col  = "created_at" if "created_at" in df.columns else "application_date"
    _ts_vals = pd.to_datetime(df[_ts_col], errors="coerce")
    _has_subday = _ts_vals.notna().any() and (_ts_vals.dropna().dt.time != pd.Timestamp("00:00:00").time()).any()
    _pbar.update(1)

    # ── 11 · rolling feature defaults (recomputed post-split in FP-3) ─────────
    # Placeholder values ensure the columns exist in the DataFrame even if FP-3
    # decides rolling computation is impossible.  Default 0.0 for counts/rates,
    # 1.0 for diversity (no diversity = 1 type).
    _pbar.set_description("rolling defaults")
    df["platform_volume_3h"]           = np.float32(0.0)
    df["platform_auto_fraction_3h"]    = np.float32(0.0)
    df["platform_content_type_div_3h"] = np.float32(1.0)
    df["platform_automation_rate"]     = np.float32(0.0)
    _pbar.update(1)

    # ── 12 · text fill ────────────────────────────────────────────────────────
    # Hopsworks does not accept NaN in string columns.  Replace missing text
    # with the empty string so downstream text vectorizers see a consistent input.
    _pbar.set_description("text fill")
    df[TEXT_COL]   = df[TEXT_COL].fillna("")
    df[TEXT_COL_2] = df[TEXT_COL_2].fillna("")
    df[TEXT_COL_3] = df[TEXT_COL_3].fillna("")
    _pbar.update(1)

    # ── 13 · diagnostics ──────────────────────────────────────────────────────
    # Print fill rates and flag prevalence so the operator can sanity-check
    # the dataset before moving to target setup.  Unexpected zeros here usually
    # mean a schema drift or a bad acquisition sample.
    _pbar.set_description("diagnostics")
    if _has_subday:
        _pbar.write(f"Sub-day timestamps in '{_ts_col}' — rolling 3h features computed in FP-3.")
    else:
        _pbar.write(f"'{_ts_col}' has no sub-day precision — rolling features default to 0/1.")
    _pbar.write(f"{TEXT_COL} fill rate:   {(df[TEXT_COL]   != '').mean():.1%}")
    _pbar.write(f"{TEXT_COL_2} fill rate: {(df[TEXT_COL_2] != '').mean():.1%}")
    _pbar.write(f"{TEXT_COL_3} fill rate: {(df[TEXT_COL_3] != '').mean():.1%}")
    _pbar.write(f"has_monetary_decision:  {df['has_monetary_decision'].mean():.1%} positive")
    _pbar.write(f"has_account_decision:   {df['has_account_decision'].mean():.1%} positive")
    _pbar.write(f"has_provision_decision: {df['has_provision_decision'].mean():.1%} positive")
    _pbar.write(f"is_multi_action:        {df['is_multi_action_decision'].mean():.1%} positive")
    _pbar.update(1)

## FP-2 · Target Setup

This stage prepares the classification target.  It strips the raw category prefix,
collapses ultra-rare classes (≤20 rows) to keep the problem tractable, builds
integer label maps, and visualises the resulting class distribution.  The label maps
are later serialised to `category_metadata.json` so the Training Pipeline can decode
predictions consistently.

**Inputs:** `df["category"]` from FP-1.  
**Outputs:** `df["target"]`, `df["target_enc"]`, `label_map`, `label_map_inv`, `target_labels`.

In [ ]:
# Strip the configured prefix from raw category strings so labels are human-readable.
df["target"] = df["category"].str.replace(PREFIX, "", regex=False)

# Count rows per class and identify any categories with 20 or fewer samples.
# These are too rare to learn reliably and are dropped to avoid singleton classes.
vc_all = df["target"].value_counts()
_rare  = vc_all[vc_all <= 20].index.tolist()
if _rare:
    print(f"Dropping {len(_rare)} class(es) with ≤20 rows: {_rare}")
    df = df[~df["target"].isin(_rare)].copy()
else:
    print("No classes with ≤20 rows — nothing dropped.")

# Build the canonical ordered list of labels and locate the catch-all OTHER class.
target_labels = sorted(df["target"].unique())
short_other   = OTHER_CAT.replace(PREFIX, "")
other_idx     = target_labels.index(short_other)

# Create forward (label → int) and inverse (int → label) mappings.
label_map     = {lbl: i for i, lbl in enumerate(target_labels)}
label_map_inv = {i: lbl for lbl, i in label_map.items()}
n_classes     = len(target_labels)

# Encode the target column as integers for model consumption.
df["target_enc"] = df["target"].map(label_map)

# Print the final class distribution so the operator sees imbalance before training.
vc = df["target"].value_counts()
print(f"{n_classes} classes  |  OTHER index = {other_idx}  ('{short_other}')")
for lbl, cnt in vc.items():
    print(f"  {lbl:<60s} {cnt:>5}  ({100*cnt/len(df):.1f}%)")

# Plot the distribution with red highlighting the catch-all OTHER class.
fig, ax = plt.subplots(figsize=(14, 5))
colors = ["#f87171" if lbl == short_other else "#6c8fff" for lbl in vc.index]
bars   = ax.bar(range(len(vc)), vc.values, color=colors)
ax.set_xticks(range(len(vc)))
ax.set_xticklabels([l.replace("_", "\n") for l in vc.index], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Count")
imb = vc.max() / vc.min()
ax.set_title(f"Category distribution  |  imbalance ratio {imb:.0f}:1  |  red = catch-all OTHER")
for bar, cnt in zip(bars, vc.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f"{cnt:,}", ha="center", fontsize=7)
plt.tight_layout(); plt.show()


## FP-3 · Feature Definition & Train / Test Split

FP-3 defines the exact feature columns that will be stored in Hopsworks, performs a
chronological train/test split (preserving temporal causality), and computes two advanced
feature families:
1. **Rolling 3-hour platform statistics** (volume, automation fraction, content-type diversity)
   computed with vectorised searchsorted over sub-day timestamps.
2. **Platform automation rate** — the historical fraction of automated decisions per platform,
   calculated from training rows only to prevent leakage.

**Inputs:** `df` from FP-1, `_has_subday` from FP-1, TOML feature lists.  
**Outputs:** `X_tr`, `X_te`, `y_tr`, `y_te`, `FEATURE_COLS`, `CATEGORICAL_FEATURES`, `NUMERICAL_FEATURES`.

In [ ]:


def _drop_na_timestamps(df, X_struct, y, _app_ts, _pbar):
    """Drop rows whose application_date could not be parsed.

    Mutates *df*, *X_struct*, *y*, and *_app_ts* in-place by filtering to
    valid timestamps, then returns the updated objects.
    """
    _n_nat = _app_ts.isna().sum()
    _pbar.write(f"INFO: dropping {_n_nat} row(s) with unparseable application_date.")
    _valid_mask = _app_ts.notna()
    df = df.loc[_valid_mask].copy()
    X_struct = X_struct.loc[_valid_mask].copy()
    y = y[_valid_mask.values]
    _app_ts = _app_ts.loc[_valid_mask]
    return df, X_struct, y, _app_ts


def _warn_singleton_classes(y, _tr_pos, label_map_inv, _pbar):
    """Emit a warning if any class has fewer than 2 training samples."""
    _class_counts = pd.Series(y[_tr_pos]).value_counts()
    _singletons = _class_counts[_class_counts < 2]
    if len(_singletons):
        _pbar.write(
            f"WARNING: {len(_singletons)} class(es) with only 1 training sample: "
            + ", ".join(label_map_inv.get(int(i), str(i)) for i in _singletons.index)
        )


def _compute_rolling_features(df, _ts_col, FEATURE_COLS, _ROLLING_FEATS,
                               _tr_idx_set, _fp, _pbar):
    """Compute 3-hour rolling volume, automation fraction, and content-type diversity.

    Runs one parallel worker per platform using ``fp3_platform_worker``.
    Mutates *df* in-place (adds ``platform_volume_3h``,
    ``platform_auto_fraction_3h``, ``platform_content_type_div_3h``).
    """
    _auto_col = (df.get("automated_detection", pd.Series("", index=df.index))
                 .fillna("").str.lower() == "yes").astype(np.float32)
    _ts_fp3 = pd.to_datetime(df[_ts_col], errors="coerce")
    _roll_df = pd.DataFrame({
        "platform_name": df["platform_name"].values,
        "_ts": _ts_fp3.values,
        "_auto": _auto_col.values,
        "_ctp": df["content_type_primary"].fillna("UNKNOWN").values,
        "_istr": df.index.isin(_tr_idx_set),
        "_pos": np.arange(len(df), dtype=np.int64),
    })
    del _auto_col, _ts_fp3

    _WIN_NS = np.int64(_fp.get("rolling_window_hours", 3) * 3600 * 1_000_000_000)
    _n_plats = _roll_df["platform_name"].nunique()

    _vol_buf = np.zeros(len(df), dtype=np.float32)
    _af_buf = np.zeros(len(df), dtype=np.float32)
    _div_buf = np.ones(len(df), dtype=np.float32)

    _FP3_WORKERS = min(_fp.get("platform_workers", 8), _n_plats)
    with ThreadPoolExecutor(max_workers=_FP3_WORKERS) as _fp3_pool:
        _fp3_futs = {
            _fp3_pool.submit(fp3_platform_worker, (pl, gr), _WIN_NS, "all_prior"): pl
            for pl, gr in _roll_df.groupby("platform_name", sort=False)
        }
        with _tqdm(total=_n_plats, desc="  platforms", unit="plat", leave=False) as _ppbar:
            for _fp3_fut in as_completed(_fp3_futs):
                _ppbar.set_description(str(_fp3_futs[_fp3_fut])[:28])
                _r = _fp3_fut.result()
                _tr_p, _vol_tr, _af_tr, _div_tr, _te_p, _vol_te, _af_te, _div_te = _r
                if len(_tr_p):
                    _vol_buf[_tr_p] = _vol_tr
                    _af_buf[_tr_p] = _af_tr
                    _div_buf[_tr_p] = _div_tr
                if len(_te_p):
                    _vol_buf[_te_p] = _vol_te
                    _af_buf[_te_p] = _af_te
                    _div_buf[_te_p] = _div_te
                _ppbar.update(1)

    df["platform_volume_3h"] = _vol_buf
    df["platform_auto_fraction_3h"] = _af_buf
    df["platform_content_type_div_3h"] = _div_buf
    del _roll_df, _vol_buf, _af_buf, _div_buf
    _pbar.write(
        f"Rolling 3h features recomputed — "
        f"{len([f for f in _ROLLING_FEATS if f in FEATURE_COLS])} cols, "
        f"{_n_plats:,} platforms, {_FP3_WORKERS} workers."
    )


def _compute_platform_automation_rate(df, FEATURE_COLS, _tr_idx, _pbar):
    """Compute per-platform automation rate from training rows only.

    Mutates *df* in-place (adds ``platform_automation_rate``).
    """
    _auto_flag = (df.get("automated_detection", pd.Series("", index=df.index))
                  .fillna("").str.lower() == "yes").astype(float)
    _train_rate = _auto_flag.loc[_tr_idx].groupby(df.loc[_tr_idx, "platform_name"]).mean()
    df["platform_automation_rate"] = df["platform_name"].map(_train_rate).fillna(np.float32(0))
    _pbar.write(f"  platform_automation_rate: mean={df['platform_automation_rate'].mean():.3f}")


def _propagate_platform_feats_to_X(X_struct, df, _tr_idx, _te_idx, _feats_in_cols):
    """Copy rolling / automation-rate feature values from *df* back into *X_struct*.

    These features are computed on the full *df* but must be propagated into the
    split-specific feature matrices.
    """
    for _f in _feats_in_cols:
        X_struct.loc[_tr_idx, _f] = df.loc[_tr_idx, _f].values
        X_struct.loc[_te_idx, _f] = df.loc[_te_idx, _f].values


# ── Config ─────────────────────────────────────────────────────────────────
# Read the exact feature lists from TOML.  Copying with [:] prevents accidental
# mutation of the original config dict.
_feat = cfg["features"]
CATEGORICAL_FEATURES = _feat["categorical"]["columns"][:]
NUMERICAL_FEATURES = _feat["numerical"]["columns"][:]

_fp = cfg["feature_pipeline"]
missing_str = _fp["missing_string"]
missing_num = _fp["missing_numeric"]

# Rolling and platform-derived features that will be recomputed in FP-3.
_ROLLING_FEATS = ["platform_volume_3h", "platform_auto_fraction_3h",
                   "platform_content_type_div_3h"]
_PLATFORM_FEATS = _ROLLING_FEATS + ["platform_automation_rate"]

_FP3_STEPS = 6
with _tqdm(total=_FP3_STEPS, desc="FP-3", unit="step") as _pbar:

    # 1 · Feature list + fill absent columns
    # Ensure every column declared in TOML exists in the DataFrame.  Missing columns
    # are filled with sentinel values so the schema stays consistent across runs.
    _pbar.set_description("feature list")
    for _c in CATEGORICAL_FEATURES:
        if _c not in df.columns:
            df[_c] = missing_str
            _pbar.write(f"  INFO: categorical '{_c}' absent — filled with {missing_str!r}")
    for _n in NUMERICAL_FEATURES:
        if _n not in df.columns:
            df[_n] = missing_num
            _pbar.write(f"  INFO: numerical '{_n}' absent — filled with {missing_num}")
    # Append platform-derived features to the numerical list if they exist in df.
    NUMERICAL_FEATURES += [f for f in _PLATFORM_FEATS
                            if f in df.columns and f not in NUMERICAL_FEATURES]
    FEATURE_COLS = CATEGORICAL_FEATURES + NUMERICAL_FEATURES
    LOW_CARD = [c for c in CATEGORICAL_FEATURES if c not in HIGH_CARD]
    _pbar.update(1)

    # 2 · Build X_struct + y
    # X_struct is the matrix of model inputs; y is the encoded target vector.
    # We coerce types now so Hopsworks receives clean Arrow-friendly columns.
    _pbar.set_description("X_struct")
    X_struct = df[FEATURE_COLS].copy()
    for _c in CATEGORICAL_FEATURES:
        X_struct[_c] = X_struct[_c].fillna(missing_str).astype(str)
    for _n in NUMERICAL_FEATURES:
        X_struct[_n] = pd.to_numeric(X_struct[_n], errors="coerce").fillna(missing_num)
    y = df["target_enc"].values
    _pbar.update(1)

    # 3 · Chronological sort + split index labels + class check
    # Sorting by application_date guarantees temporal causality: training rows are
    # strictly older than test rows, preventing future-information leakage.
    # We defer creating X_tr/X_te until step 6 to avoid an expensive extra copy.
    _pbar.set_description("sort & split")
    _app_ts = pd.to_datetime(df["application_date"], errors="coerce")
    _n_nat = _app_ts.isna().sum()
    if _n_nat > 0:
        df, X_struct, y, _app_ts = _drop_na_timestamps(
            df, X_struct, y, _app_ts, _pbar
        )
    _sorted_pos = _app_ts.argsort().values
    _n_tr = int(len(_sorted_pos) * (1 - _fp["test_size"]))
    _tr_pos = _sorted_pos[:_n_tr]
    _te_pos = _sorted_pos[_n_tr:]
    _tr_idx = X_struct.index[_tr_pos]
    _te_idx = X_struct.index[_te_pos]
    _tr_idx_set = set(_tr_idx)

    _warn_singleton_classes(y, _tr_pos, label_map_inv, _pbar)
    _pbar.update(1)

    # 4 · Rolling features — parallel platform workers, fully vectorised
    #
    # All three metrics use searchsorted + prefix-sum arrays (no Python
    # callbacks, no pandas rolling.apply).  Platforms are independent so they
    # run in a ThreadPoolExecutor; write-backs are buffered into numpy arrays
    # and flushed once per column at the end.
    _pbar.set_description("rolling features")
    _has_rolling = any(f in FEATURE_COLS for f in _ROLLING_FEATS)
    if _has_rolling and _has_subday and "platform_name" in df.columns:
        _compute_rolling_features(df, _ts_col, FEATURE_COLS, _ROLLING_FEATS,
                                   _tr_idx_set, _fp, _pbar)
    _pbar.update(1)

    # 5 · platform_automation_rate from training rows only
    # Using only training rows prevents target leakage: the test set must not
    # influence any feature value.
    _pbar.set_description("automation rate")
    if "platform_automation_rate" in FEATURE_COLS and "platform_name" in df.columns:
        _compute_platform_automation_rate(df, FEATURE_COLS, _tr_idx, _pbar)
    _pbar.update(1)

    # 6 · Propagate rolling into X_struct -> create X_tr / X_te (single copy each)
    # After rolling features are computed on the full df, copy them into X_struct
    # and then slice out the train/test matrices.  Doing this in one shot minimises
    # peak memory usage.
    _pbar.set_description("propagate & split")
    _feats_in_cols = [f for f in _PLATFORM_FEATS if f in FEATURE_COLS]
    if _feats_in_cols:
        _propagate_platform_feats_to_X(X_struct, df, _tr_idx, _te_idx, _feats_in_cols)
    X_tr = X_struct.loc[_tr_idx].copy()
    X_te = X_struct.loc[_te_idx].copy()
    y_tr = y[_tr_pos]
    y_te = y[_te_pos]
    _pbar.update(1)

print(f"EDA split (chronological, not persisted):  Train {len(X_tr):,}  Test {len(X_te):,}")
print(f"Structured features ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"  Categorical ({len(CATEGORICAL_FEATURES)}): {CATEGORICAL_FEATURES}")
print(f"  Numerical   ({len(NUMERICAL_FEATURES)}): {NUMERICAL_FEATURES}")
print(f"Smallest class in train: {np.bincount(y_tr.astype(int)).min()}")




## FP-4 · Upload Features to Hopsworks Feature Store

This stage materialises the engineered features in Hopsworks.  It creates four
Feature Groups:

1. **`category_features`** (structured) — primary key `row_id`, holds all categorical/numerical
   features plus `target_enc`.
2. **`category_features_text_1`** — holds `row_id` + `incompatible_content_ground`.
3. **`category_features_text_2`** — holds `row_id` + `incompatible_content_explanation`.
4. **`category_features_text_3`** — holds `row_id` + `decision_facts`.

Each existing feature group is deleted first so the new run always starts from a clean
schema.  Text groups are recreated even if they already exist, because `row_id` values
change when the underlying dataset is re-acquired; skipping reinsertion would cause NULLs
on the left join.

**Inputs:** `X_struct`, `y`, `df`, TOML Hopsworks config.  
**Outputs:** four ingested Feature Groups in Hopsworks.

In [ ]:
# ── row_id: use source UUID directly — guaranteed unique per DSA record ────────
# The DSA dataset provides a uuid column that is naturally unique.  Using it as the
# primary key ensures stable joins across feature groups and avoids synthetic key drift.
if "uuid" in df.columns:
    _row_ids = df.loc[X_struct.index, "uuid"].astype(str).tolist()
else:
    # Fallback: sequential integer as string — collision-proof and stable within the run.
    _row_ids = np.arange(len(X_struct), dtype=np.int64).astype(str).tolist()

# Verify uniqueness; duplicates would break the primary-key constraint in Hopsworks.
_n_dupes = pd.Series(_row_ids).duplicated().sum()
if _n_dupes:
    print(f"WARNING: {_n_dupes:,} duplicate row_id values — check uuid column for collisions.")
else:
    print(f"row_id: all unique  ({len(_row_ids):,} rows)")

# ── Struct DataFrame ─────────────────────────────────────────────────────────────
# Assemble the structured feature matrix that will become the main Feature Group.
df_struct = X_struct.copy()
df_struct.insert(0, "row_id", _row_ids)
df_struct["target_enc"] = y

# If an event-time column is configured, parse it to datetime so Hopsworks recognises it.
_fgcfg  = cfg["hopsworks"]["feature_group"]
_et_col = _fgcfg.get("event_time_col")
if _et_col and _et_col in df.columns:
    df_struct[_et_col] = pd.to_datetime(
        df.loc[X_struct.index, _et_col], errors="coerce"
    ).values

# Hopsworks' Kafka online-store writer (Avro serialisation) does not support
# pandas nullable extension integer dtypes (Int16, Int8, Int32, etc.).  Since
# all nulls were already filled with missing_num in FP-3, we can safely cast
# any remaining extension integer columns to standard numpy int64.
for _col in df_struct.columns:
    if pd.api.types.is_extension_array_dtype(df_struct[_col]) and pd.api.types.is_integer_dtype(df_struct[_col]):
        df_struct[_col] = df_struct[_col].astype("int64")

# Delete the feature view first (it references the struct FG; must go before the FG).
_fv_cfg = cfg["hopsworks"].get("feature_view", {})
_fv_name_del = _fv_cfg.get("name", FV_NAME)
_fv_ver_del  = _fv_cfg.get("version", FV_VERSION)
try:
    _existing_fv = fs.get_feature_view(name=_fv_name_del, version=_fv_ver_del)
    _existing_fv.delete_all_training_datasets()
    _existing_fv.delete()
    print(f"Feature View '{_fv_name_del}' v{_fv_ver_del}: deleted.")
except Exception:
    pass

# Delete the struct FG unconditionally so we always start from a clean Hudi table.
try:
    _existing_struct = fs.get_feature_group(name=FG_NAME, version=FG_VERSION)
    _existing_struct.delete()
    print(f"Struct FG '{FG_NAME}' v{FG_VERSION}: deleted (will be recreated).")
except Exception:
    pass

# Recreate the structured Feature Group with the same name/version and insert data.
fg_struct = fs.get_or_create_feature_group(
    name=FG_NAME,
    version=FG_VERSION,
    primary_key=["row_id"],
    event_time=_et_col,
    description=_fgcfg.get(
        "description",
        "Category classification structured features. Joined to the text FG via row_id.",
    ),
    online_enabled=_fgcfg.get("online_enabled", False),
)
fg_struct.insert(df_struct.drop_duplicates(subset=["row_id"]), write_options={"wait_for_job": True})
print(f"Struct FG '{FG_NAME}' v{FG_VERSION}: {len(df_struct):,} rows ingested.")

# ── Text Feature Groups — always delete and reinsert to match current struct FG row_ids ──
# Skipping reinsert when a text FG already exists causes NULL text columns when the
# struct FG is recreated with a new dataset (row_ids change; left-join returns NULLs).
for _fg_txt_name, _fg_txt_ver, _fg_txt_cfg_key, _txt_col, _df_txt in [
    (FG_TEXT_1_NAME, FG_TEXT_1_VERSION, "feature_group_text_1", TEXT_COL,
     pd.DataFrame({"row_id": _row_ids,
                   TEXT_COL: df.loc[X_struct.index, TEXT_COL].fillna("").values})),
    (FG_TEXT_2_NAME, FG_TEXT_2_VERSION, "feature_group_text_2", TEXT_COL_2,
     pd.DataFrame({"row_id": _row_ids,
                   TEXT_COL_2: df.loc[X_struct.index, TEXT_COL_2].fillna("").values})),
    (FG_TEXT_3_NAME, FG_TEXT_3_VERSION, "feature_group_text_3", TEXT_COL_3,
     pd.DataFrame({"row_id": _row_ids,
                   TEXT_COL_3: df.loc[X_struct.index, TEXT_COL_3].fillna("").values})),
]:
    _fg_txt_cfg = cfg["hopsworks"].get(_fg_txt_cfg_key, {})
    try:
        _existing_txt = fs.get_feature_group(name=_fg_txt_name, version=_fg_txt_ver)
        _existing_txt.delete()
        print(f"Text FG '{_fg_txt_name}' v{_fg_txt_ver}: deleted (recreating to match struct FG).")
    except Exception:
        pass
    _fg_txt = fs.get_or_create_feature_group(
        name=_fg_txt_name,
        version=_fg_txt_ver,
        primary_key=["row_id"],
        description=_fg_txt_cfg.get("description", f"Text column '{_txt_col}' for category classification"),
        online_enabled=_fg_txt_cfg.get("online_enabled", True),
    )
    _fg_txt.insert(_df_txt.drop_duplicates(subset=["row_id"]), write_options={"wait_for_job": True})
    print(f"Text FG '{_fg_txt_name}' v{_fg_txt_ver}: {len(_df_txt):,} rows ingested.")
fg_text_1 = fs.get_feature_group(name=FG_TEXT_1_NAME, version=FG_TEXT_1_VERSION)
fg_text_2 = fs.get_feature_group(name=FG_TEXT_2_NAME, version=FG_TEXT_2_VERSION)
fg_text_3 = fs.get_feature_group(name=FG_TEXT_3_NAME, version=FG_TEXT_3_VERSION)

## FP-5 · Create Feature View

A **Feature View** is a named, versioned query over one or more Feature Groups. It is
the contract between the Feature Store and the Training Pipeline — it defines which
columns are features (X) and which is the label (y).

| Role | Column |
|---|---|
| Features (X) | All FG columns except `row_id` and `target_enc` |
| Label (y) | `target_enc` |

`_t.ipynb` calls `fv.create_train_test_split()` on this view to materialise a versioned,
reproducible train/test split in Hopsworks storage. Setting `TD_VERSION` in TP-0 pins
the split so hyperparameter searches and re-runs use identical data.

In [ ]:
# ── Create or retrieve Feature View (joins struct FG + 3 text FGs on row_id) ────
# Build the query object that defines how the four feature groups are joined.
try:
    _struct_query = fg_struct.select_except(["row_id"])
    _fv_query = (
        _struct_query
        .join(fg_text_1.select([TEXT_COL]),   on=["row_id"], join_type="left")
        .join(fg_text_2.select([TEXT_COL_2]), on=["row_id"], join_type="left")
        .join(fg_text_3.select([TEXT_COL_3]), on=["row_id"], join_type="left")
    )
except AttributeError:
    # Fallback for older SDK versions that don't support select_except:
    # manually list every struct column except row_id.
    _struct_cols  = [c for c in df_struct.columns if c != "row_id"]
    _fv_query = (
        fg_struct.select(_struct_cols)
        .join(fg_text_1.select([TEXT_COL]),   on=["row_id"], join_type="left")
        .join(fg_text_2.select([TEXT_COL_2]), on=["row_id"], join_type="left")
        .join(fg_text_3.select([TEXT_COL_3]), on=["row_id"], join_type="left")
    )

_fv_desc = cfg.get("hopsworks", {}).get("feature_view", {}).get(
    "description", "Category features view for Hopsworks-managed train/test splitting"
)

# Register the query as a named, versioned Feature View — the contract consumed by
# the Training Pipeline when it calls fv.create_train_test_split().
fv = fs.get_or_create_feature_view(
    name=FV_NAME,
    version=FV_VERSION,
    query=_fv_query,
    description=_fv_desc,
)
print(f"Feature View '{FV_NAME}' v{FV_VERSION} ready.")
print(f"  Struct FG  : '{FG_NAME}' v{FG_VERSION}  ({len(FEATURE_COLS)} features)")
print(f"  Text FG 1  : '{FG_TEXT_1_NAME}' v{FG_TEXT_1_VERSION}  ({TEXT_COL!r})")
print(f"  Text FG 2  : '{FG_TEXT_2_NAME}' v{FG_TEXT_2_VERSION}  ({TEXT_COL_2!r})")
print(f"  Text FG 3  : '{FG_TEXT_3_NAME}' v{FG_TEXT_3_VERSION}  ({TEXT_COL_3!r})")
print(f"  Join       : left join on row_id (×3)")
print(f"  Label      : target_enc")
print()
print("Next step: run _t.ipynb → TP-0 will call fv.create_train_test_split() to")
print("materialise the authoritative train/test split in Hopsworks storage.")

## FP-6 · Save Pipeline Metadata

The final stage serialises every label map, feature list, and Hopsworks entity name
to a local JSON file (`category_metadata.json`).  The Training Pipeline reads this file
so it knows the exact class ordering, which columns are categorical vs numerical, and
which Feature View to query.  Keeping metadata out of the notebook makes re-runs and
CI/CD deterministic.

**Inputs:** all variables produced in FP-2, FP-3, and FP-4.  
**Outputs:** `category_metadata.json` on local disk.

In [ ]:
# Assemble a single dict containing everything the Training Pipeline needs to know
# about the feature schema, label encoding, and Hopsworks entity names.
metadata = {
    "target_labels":        target_labels,
    "label_map":            label_map,
    "label_map_inv":        {str(k): v for k, v in label_map_inv.items()},
    "other_idx":            int(other_idx),
    "n_classes":            int(n_classes),
    "short_other":          short_other,
    "FEATURE_COLS":         FEATURE_COLS,
    "CATEGORICAL_FEATURES": CATEGORICAL_FEATURES,
    "NUMERICAL_FEATURES":   NUMERICAL_FEATURES,
    "HIGH_CARD":            HIGH_CARD,
    "LOW_CARD":             LOW_CARD,
    "TEXT_COL":             TEXT_COL,
    "TEXT_COL_2":           TEXT_COL_2,
    "TEXT_COL_3":           TEXT_COL_3,
    "fg_name":              FG_NAME,
    "fg_version":           FG_VERSION,
    "fg_text_1_name":       FG_TEXT_1_NAME,
    "fg_text_1_version":    FG_TEXT_1_VERSION,
    "fg_text_2_name":       FG_TEXT_2_NAME,
    "fg_text_2_version":    FG_TEXT_2_VERSION,
    "fg_text_3_name":       FG_TEXT_3_NAME,
    "fg_text_3_version":    FG_TEXT_3_VERSION,
    "fv_name":              FV_NAME,
    "fv_version":           FV_VERSION,
}

# Write with indentation so diffs in version control are human-readable.
with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata saved → {METADATA_PATH}")
print(f"  Classes: {n_classes}  |  Features: {len(FEATURE_COLS)}")
print(f"  Text columns: {TEXT_COL!r}, {TEXT_COL_2!r}, {TEXT_COL_3!r}")
print(f"  Struct FG    : '{FG_NAME}' v{FG_VERSION}")
print(f"  Text FG 1    : '{FG_TEXT_1_NAME}' v{FG_TEXT_1_VERSION}")
print(f"  Text FG 2    : '{FG_TEXT_2_NAME}' v{FG_TEXT_2_VERSION}")
print(f"  Text FG 3    : '{FG_TEXT_3_NAME}' v{FG_TEXT_3_VERSION}")
print(f"  Feature View : '{FV_NAME}' v{FV_VERSION}")